In [6]:
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import torch
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
# Erstmal schauen wie der Tokenizer die Marker kodiert
test = tokenizer("« publication price»", return_tensors="pt")
print("Token IDs:", test['input_ids'])
print("Tokens:   ", tokenizer.convert_ids_to_tokens(test['input_ids'][0]))

# Und direkt im Input nachschauen
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print("\nAlle Tokens:")
for i, t in enumerate(tokens):
    print(f"  {i:3d}: {repr(t)}")

NameError: name 'tokenizer' is not defined

In [5]:
# Statt einzelner Token-ID: alle Tokens durchsuchen
open_positions  = [i for i, t in enumerate(tokens) if '«' in t]
close_positions = [i for i, t in enumerate(tokens) if '»' in t]

print(f"« Positionen: {open_positions}")
print(f"» Positionen: {close_positions}")

« Positionen: [87, 94, 101, 106, 113, 118]
» Positionen: [92, 99, 104, 111, 116, 121]


In [5]:
# 1. Modell laden mit 4-bit Quantisierung
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
model = AutoModel.from_pretrained(
    "deepseek-ai/deepseek-coder-6.7b-base",
    quantization_config=bnb_config,
    output_hidden_states=True,
    device_map="auto"
)
model.eval()

# 2. Input formatieren (wie in Figure 3 des Papers)
schema = """
CREATE TABLE publication (
    publication_id NUMBER PRIMARY KEY,
    book_id NUMBER,
    price NUMBER
);
CREATE TABLE book (
    book_id NUMBER PRIMARY KEY,
    title TEXT,
    writer TEXT
);
"""

question = "Show the titles of books in descending order of publication price."

# Kandidaten-Spalten mit « » Markern (wie im Paper)
candidates = [
    ("publication", "publication_id"),
    ("publication", "book_id"),
    ("publication", "price"),
    ("book", "book_id"),
    ("book", "title"),
    ("book", "writer"),
]

candidate_str = "\n".join([f"« {t} {c}»" for t, c in candidates])

prompt = f"{schema}\nTo answer: {question}\nWe need columns:\n{candidate_str}"
print("=== INPUT ===")
print(prompt)

# 3. Tokenisieren
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(f"\n=== TOKEN ANZAHL: {inputs['input_ids'].shape[1]} ===")

# 4. Forward Pass - Hidden States extrahieren
with torch.no_grad():
    outputs = model(**inputs)

# Letzter Hidden State: Shape [1, seq_len, hidden_size]
last_hidden = outputs.hidden_states[-1]
print(f"\n=== HIDDEN STATE SHAPE: {last_hidden.shape} ===")
# z.B. [1, 156, 4096] für 6.7B Modell

# 5. Positionen der « und » Token finden
# Die Marker « und » werden vom Tokenizer als UTF-8 Bytes kodiert: 'Â«' und 'Â»'
# Deshalb suchen wir nach der Byte-Darstellung statt dem Unicode-Zeichen
# Debug: Tokens um Position 87 anzeigen
print(tokens[85:95])
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

open_positions  = [i for i, t in enumerate(tokens) if t == 'Â«']
close_positions = [i for i, t in enumerate(tokens) if t == 'Â»']

print(f"\n=== MARKER POSITIONEN ===")
print(f"« gefunden: {len(open_positions)}x  » gefunden: {len(close_positions)}x")

# 6. Hidden Vektoren der Marker extrahieren und konkatenieren
# Das ist der Kern des Papers: E_alpha ⊕ E_omega
pair_vectors = []
for alpha, omega in zip(open_positions, close_positions):
    e_alpha = last_hidden[0, alpha, :]   # hidden state bei «
    e_omega = last_hidden[0, omega, :]   # hidden state bei »
    combined = torch.cat([e_alpha, e_omega], dim=-1)  # [hidden*2]
    pair_vectors.append(combined)

C = torch.stack(pair_vectors)  # [num_candidates, hidden*2]
print(f"\n=== KANDIDATEN-MATRIX C: {C.shape} ===")
# z.B. [6, 8192] für 6.7B (4096*2)

# 7. Linear Layer (noch UNTRAINIERT - zeigt nur die Struktur)
# Nach dem Fine-tuning würde hier die trainierte w_relevance Matrix stehen
hidden_size = last_hidden.shape[-1]
w_relevance = torch.nn.Linear(hidden_size * 2, 1).to(C.device, dtype=C.dtype)

with torch.no_grad():
    logits = w_relevance(C).squeeze(-1)  # [num_candidates]
    probs  = torch.sigmoid(logits)

print(f"\n=== RELEVANZ-WAHRSCHEINLICHKEITEN (untrainiert = zufällig) ===")
for (table, col), prob in zip(candidates, probs):
    print(f"  {table}.{col:20s}  p={prob.item():.3f}")

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [1]:
import torch
print(f"Gesamt VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Belegt:      {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Frei:        {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.2f} GB")

Gesamt VRAM: 15.8 GB
Belegt:      0.00 GB
Frei:        15.77 GB
